🧠 AutoGen Assessment Scenario
🎯 Title: Multi-Agent AI System for Automated Code Review & Execution
🔹 Scenario Overview

You are part of an AI engineering team building an AutoGen-based developer assistant.

The system should:

Accept a user problem statement
Generate code
Review the code
Execute it
Provide feedback and corrections

This must be implemented using multiple collaborating agents.

🔹 Problem Statement

A user gives the following input:

“Write a Python function to find the second largest number in a list and handle edge cases.”

Your AutoGen system should:

Generate correct code
Validate logic
Execute the code
Handle errors
Provide final response
🔹 Requirements
✅ 1. Agent Design (MANDATORY)

Create at least 3 agents:

🔸 Coder Agent
Generates Python code
Should follow best practices
Must handle edge cases
🔸 Reviewer Agent
Reviews code for:
Bugs
Logic errors
Edge cases
Suggests improvements
🔸 Executor Agent (UserProxy)
Runs the code
Returns output/errors
🔹 2. Agent Collaboration Flow

Design the flow such that:

User → Coder → Reviewer → Executor → Reviewer (if error) → Final Output

👉 The system should not stop after one step
👉 It must iterate until correct output is achieved

🔹 3. Functional Expectations

Your system must:

✔ Handle incorrect code generated initially
✔ Ensure reviewer gives structured feedback
✔ Prevent infinite loops
✔ Show final clean output

🔹 4. Technical Constraints
Use AutoGen framework
Use LLM-based agents
Enable code execution
Add termination condition
🔹 5. Edge Cases to Handle

Test your system with:

Empty list
List with one element
Duplicate values
Negative numbers

In [1]:
!pip uninstall -y autogen pyautogen autogen-agentchat autogen-core autogen-ext numpy
!pip install --no-cache-dir numpy==1.26.4 pyautogen==0.2.25 openai

Found existing installation: pyautogen 0.2.25
Uninstalling pyautogen-0.2.25:
  Successfully uninstalled pyautogen-0.2.25
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 248.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.1/257.1 kB 103.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1

In [1]:
import autogen
import numpy as np

print("autogen ok")
print("numpy version:", np.__version__)

autogen ok
numpy version: 1.26.4


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [2]:
# ============================================================
# Multi-Agent AI System for Automated Code Review & Execution
# Assessment-Ready AutoGen Architecture
# ============================================================

import os
import autogen

try:
    from google.colab import userdata
    secret_key = userdata.get("GROQ_API_KEY")
    if secret_key:
        os.environ["GROQ_API_KEY"] = secret_key
except Exception:
    pass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = "gsk_your_key_here"

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
if not GROQ_API_KEY or GROQ_API_KEY == "gsk_your_key_here":
    raise ValueError("Set a valid GROQ_API_KEY in Colab Secrets or directly in the code.")

config_list = [
    {
        "model": "llama-3.3-70b-versatile",
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "api_type": "openai",
    }
]

llm_config = {
    "config_list": config_list,
    "temperature": 0,
    "timeout": 120,
}

problem_statement = """
User Problem:
Write a Python function to find the second largest number in a list and handle edge cases.

Mandatory requirements:
1. Return the second largest distinct number.
2. Handle:
   - empty list
   - list with one element
   - duplicate values
   - negative numbers
3. Include test cases.
4. If code fails, fix it and retry.
5. Reviewer must only approve after execution succeeds.
6. Final approved response must clearly show the clean solution and expected outputs.
"""

coder = autogen.AssistantAgent(
    name="Coder",
    llm_config=llm_config,
    system_message="""
You are the Coder Agent.

Your responsibilities:
- Write clean, production-style Python code.
- Solve the task exactly as requested.
- Handle all edge cases explicitly.
- Include executable tests in the same code block.
- When sending code, ALWAYS send one complete Python code block.
- Do not ask the user to edit code manually.
- If reviewer/executor reports an issue, fix it in the next turn.
- When the solution is stable, provide a final polished explanation.

Coding standards:
- Use clear function names
- Add comments only where useful
- Follow PEP8
- Avoid unnecessary dependencies
""",
)

reviewer = autogen.AssistantAgent(
    name="Reviewer",
    llm_config=llm_config,
    system_message="""
You are the Reviewer Agent.

Your responsibilities:
- Review code for correctness, robustness, and edge cases.
- Check whether the code returns the second largest DISTINCT value.
- Verify behavior for:
  * []
  * [x]
  * duplicate-heavy input
  * negative numbers
- Inspect execution output from Executor.
- If something is wrong, give crisp feedback prefixed with FIX:
- If code is ready for execution, instruct execution prefixed with EXECUTE:
- Approve only when logic and execution both pass.
- Final approval must be prefixed with APPROVED:

Rules:
- Be strict.
- Never approve without runtime evidence.
- Keep feedback structured and short.
""",
)

executor = autogen.UserProxyAgent(
    name="Executor",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=8,
    code_execution_config={
        "work_dir": "autogen_code_review_workspace",
        "use_docker": False,
    },
    llm_config=False,
    system_message="""
You are the Executor Agent.

Responsibilities:
- Execute Python code blocks exactly as provided.
- Return runtime output or traceback.
- Do not rewrite code.
- If there is no code block to run, respond briefly with what was received.
""",
    is_termination_msg=lambda msg: "APPROVED:" in msg.get("content", ""),
)

allowed_transitions = {
    executor: [coder],
    coder: [reviewer],
    reviewer: [executor, coder],
}

groupchat = autogen.GroupChat(
    agents=[executor, coder, reviewer],
    messages=[],
    max_round=12,
    speaker_transitions_type="allowed",
    allowed_or_disallowed_speaker_transitions=allowed_transitions,
)

manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config,
)

initial_message = f"""
Act as a top-tier AI engineering code-review system.

Workflow you must follow:
1. Coder writes code.
2. Reviewer checks logic.
3. Reviewer sends to Executor if ready.
4. Executor runs code and returns output/errors.
5. Reviewer either requests fixes or approves.
6. Iterate until approved or max rounds reached.

Task:
{problem_statement}

Expected final outcome:
- correct Python function
- tests covering all required edge cases
- clean final explanation
- reviewer approval
"""

chat_result = executor.initiate_chat(
    manager,
    message=initial_message,
)

print("\n" + "=" * 80)
print("FINAL CHAT SUMMARY")
print("=" * 80)

for i, msg in enumerate(groupchat.messages, start=1):
    role = msg.get("name", msg.get("role", "unknown"))
    content = msg.get("content", "")
    print(f"\n[{i}] {role}")
    print("-" * 60)
    print(content)

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

Executor (to chat_manager):


Act as a top-tier AI engineering code-review system.

Workflow you must follow:
1. Coder writes code.
2. Reviewer checks logic.
3. Reviewer sends to Executor if ready.
4. Executor runs code and returns output/errors.
5. Reviewer either requests fixes or approves.
6. Iterate until approved or max rounds reached.

Task:

User Problem:
Write a Python function to find the second largest number in a list and handle edge cases.

Mandatory requirements:
1. Return the second largest distinct number.
2. Handle:
   - empty list
   - list with one element
   - duplicate values
   - negative numbers
3. Include test cases.
4. If code fails, fix it and retry.
5. Reviewer must only approve after execution succeeds.
6. Final approved response must clearly show the clean solution and expected outputs.


Expected final outcome:
- correct Python function
- tests covering all required edge cases
- clean final explanation
- reviewer approval


---------------------------------